In [578]:
# Loading the data
import json
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.covariance import MinCovDet
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load data/valencia_cf_1/matches.json
matches_path = project_root / "tests" / "fixtures" / "testing_data" / "data" / "valencia_cf_1" / "matches.json"
raw_matches = json.load(open(matches_path))
print(f"Loaded {len(raw_matches)} matches from {matches_path}")

Loaded 100 matches from C:\Users\kazik\projects\Gaffers-Clipboard\tests\fixtures\testing_data\data\valencia_cf_1\matches.json


In [579]:
normalized_df = pd.json_normalize(
    raw_matches,
    record_path="player_performances",
    meta=[
        "id", ["data", "half_length"], 
        ["data", "home_team_name"], 
        ["data", "away_team_name"],
        ["data", "home_stats", "xg"],
        ["data", "away_stats", "xg"],
        ["data", "home_score"],
        ["data", "away_score"],
    ]
)

normalized_df["team_xg"] = float("Nan")
normalized_df["xg_against"] = float("nan")
normalized_df["goals_conceded"] = float("nan")

for idx, row in normalized_df.iterrows():
    if row["data.home_team_name"] == "Valencia CF":
        normalized_df.at[idx, "xg_against"] = row["data.away_stats.xg"]
        normalized_df.at[idx, "goals_conceded"] = row["data.away_score"]
        normalized_df.at[idx, "team_xg"] = row["data.home_stats.xg"]
    else:
        normalized_df.at[idx, "xg_against"] = row["data.home_stats.xg"]
        normalized_df.at[idx, "goals_conceded"] = row["data.home_score"]
        normalized_df.at[idx, "team_xg"] = row["data.away_stats.xg"]

normalized_df = normalized_df.drop(columns=["data.home_team_name", "data.away_team_name", "data.home_stats.xg", "data.away_stats.xg", "data.home_score", "data.away_score"])

normalized_df = normalized_df.rename(columns={
    "data.half_length": "half_length",
    "id": "match_id",
})

# safer GK filter (handles lists, strings, and NaN)
def is_gk(val):
    if pd.isna(val):
        return False
    if isinstance(val, list):
        return any(item == "GK" for item in val)
    return val == "GK"

pos_df = normalized_df[normalized_df["performance_type"].apply(is_gk)]

# drop columns that are all NaN
pos_df = pos_df.dropna(axis=1, how="all")

# safe drop (no KeyError if column already removed)
pos_df = pos_df.drop(columns=["performance_type"], errors="ignore")

cols = pos_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("match_id")))
pos_df = pos_df[cols]

print(f"Number of rows: {pos_df.shape[0]}")
print(f"Number of columns: {pos_df.shape[1]}")
print("Columns:")
for col in cols:
    print(f" - {col}")

# Output the max and min of every column, to get a sense of the range of values
for col in pos_df.columns:
    if col in ["match_id", "player_id"]:
        continue
    if pos_df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={pos_df[col].min()}, max={pos_df[col].max()}")

Number of rows: 100
Number of columns: 16
Columns:
 - match_id
 - shots_against
 - shots_on_target
 - saves
 - goals_conceded
 - save_success_rate
 - punch_saves
 - rush_saves
 - penalty_saves
 - penalty_goals_conceded
 - shoot_out_saves
 - shoot_out_goals_conceded
 - player_id
 - half_length
 - team_xg
 - xg_against
shots_against: min=0.0, max=19.0
shots_on_target: min=0.0, max=11.0
saves: min=0.0, max=11.0
goals_conceded: min=0.0, max=4.0
save_success_rate: min=0.0, max=100.0
punch_saves: min=0.0, max=2.0
rush_saves: min=0.0, max=2.0
penalty_saves: min=0.0, max=1.0
penalty_goals_conceded: min=0.0, max=1.0
shoot_out_saves: min=0.0, max=1.0
shoot_out_goals_conceded: min=0.0, max=8.0
team_xg: min=0.7, max=13.2
xg_against: min=0.0, max=4.0


In [580]:
# 1. Select every column EXCEPT your identifiers
cols_to_convert = pos_df.columns.drop(['match_id', 'player_id'])

# 2. Force them all to numeric in one swoop
pos_df[cols_to_convert] = pos_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
pos_df[cols_to_convert] = pos_df[cols_to_convert].fillna(0)

In [581]:
h_base = 10.0
vol_cols = ['shots_against', 'shots_on_target', 'saves', 'goals_conceded', 'team_xg', 'xg_against']

for col in vol_cols:
    # Scale volume metrics so a 4-min half translates to a 10-min equivalent
    pos_df[f"{col}"] = pos_df[col] * (h_base / pos_df["half_length"])

In [582]:
pos_df["xgp"] = pos_df["xg_against"] - pos_df["goals_conceded"]

In [583]:
# Cell 33a: Calculate the Absolute Heuristic (Base Performance Value)
pos_df["gk_heuristic"] = (pos_df["xgp"] * 1.5) + (np.log(pos_df["saves"] + 1.0) * (pos_df["save_success_rate"] / 100.0))

# Print the population mean and std to observe the positive skew
h_mean = pos_df["gk_heuristic"].mean()
h_std = pos_df["gk_heuristic"].std()
print(f"GK Heuristic - Mean: {h_mean:.3f}, StdDev: {h_std:.3f}")

GK Heuristic - Mean: 2.101, StdDev: 1.703


In [ ]:
# # Saving the mean and std in position_means_stds.json, in the format {"GK": {"mean": h_mean, "std": h_std}}
# means_stds_path = project_root / "workshop" / "position_means_stds.json"
# gk_means_stds = {
#     "GK": {
#         "mean": h_mean,
#         "std": h_std
#     }
# }

# if means_stds_path.exists():
#     with open(means_stds_path, "r") as f:
#         position_means_stds = json.load(f)
# else:
#     position_means_stds = {}

# position_means_stds["GK"] = gk_means_stds["GK"]

# with open(means_stds_path, "w") as f:
#     json.dump(position_means_stds, f, indent=4)

In [585]:
# Cell 33b: Standardize to a Z-Score (The True raw_score)
# By standardizing, an average performance becomes 0.0, aligning perfectly with the Sigmoid s_0 baseline.
pos_df["raw_score"] = (pos_df["gk_heuristic"] - h_mean) / h_std

# Optional: You can apply a slight multiplier here if you want GK ratings to be slightly more volatile 
# (e.g., pos_df["raw_score"] = pos_df["raw_score"] * 1.1)

In [586]:
# ---------------------------------------------------------
# The Spectator Exemption
# If a GK faces 0 shots and concedes 0 goals, force their Z-score to 0.0.
# This prevents them from being punished for a lack of save volume.
# ---------------------------------------------------------
# A. Spectator Exemption (Now uses shots_on_target instead of shots_against)
spectator_mask = (pos_df['shots_on_target'] <= 0) & (pos_df['goals_conceded'] == 0)
pos_df['raw_score'] = np.where(spectator_mask, 0.0, pos_df['raw_score'])

# B. The 1-Goal Narrow Miss (Fixes Match 24)
# Conceded 1 goal, but xGP is >= -0.25. Floor Z-Score at -0.15 (~5.7 base rating).
narrow_miss_mask = (pos_df['goals_conceded'] == 1) & (pos_df['xgp'] >= -0.25)
pos_df['raw_score'] = np.where(narrow_miss_mask & (pos_df['raw_score'] < -0.15), -0.15, pos_df['raw_score'])

# C. Positive Impact Anchor
# Prevented expected goals, conceded 2 or less. Floor Z-Score at 0.0 (~6.0 base rating).
pos_impact_mask = (pos_df['xgp'] >= 0) & (pos_df['goals_conceded'] <= 2)
pos_df['raw_score'] = np.where(pos_impact_mask & (pos_df['raw_score'] < 0.0), 0.0, pos_df['raw_score'])

# D. The Shootout Victim (Fixes Match 86)
# Conceded 3+ goals, but xGP is >= -0.50 (defense collapsed). Floor Z-score at -0.35 (~5.2 base rating).
shootout_victim_mask = (pos_df['goals_conceded'] >= 3) & (pos_df['xgp'] >= -0.50)
pos_df['raw_score'] = np.where(shootout_victim_mask & (pos_df['raw_score'] < -0.35), -0.35, pos_df['raw_score'])

# ---------------------------------------------------------
# 2. Low-Volume Forgiveness (The Match 69 Fix)
# If a GK faces <= 3 shots and concedes <= 1 goal, their rating 
# should never drop below a -0.5 Z-Score (roughly a 5.5 base rating).
# ---------------------------------------------------------
low_vol_mask = (pos_df['shots_against'] <= 3) & (pos_df['goals_conceded'] <= 1)
# Only apply the floor if their current score is worse than -0.5
pos_df['raw_score'] = np.where(low_vol_mask & (pos_df['raw_score'] < -0.5), -0.5, pos_df['raw_score'])

# If GC <= 2: Cap the punishment at Z = -0.8 (Limits the floor to ~4.7)
gc_2_mask = pos_df['goals_conceded'] <= 2
pos_df['raw_score'] = np.where(gc_2_mask & (pos_df['raw_score'] < -0.8), -0.8, pos_df['raw_score'])

# If GC == 3: Cap the punishment at Z = -1.1 (Limits the floor to ~4.0)
gc_3_mask = pos_df['goals_conceded'] == 3
pos_df['raw_score'] = np.where(gc_3_mask & (pos_df['raw_score'] < -0.95), -0.95, pos_df['raw_score'])

# ---------------------------------------------------------
# The Terminal Floor (The 4+ Goal Backstop)
# Prevents the Z-score from dropping past -1.25
# ---------------------------------------------------------
pos_df['raw_score'] = np.clip(pos_df['raw_score'], a_min=-1.25, a_max=None)

# ---------------------------------------------------------
# 3. Additive Bonuses (Penalties & Reliable Shifts)
# ---------------------------------------------------------
# A. Penalty Heroics (The Asymmetric Bonus)
total_penalty_saves = pos_df['penalty_saves'] + pos_df['shoot_out_saves']
pos_df['raw_score'] += (total_penalty_saves * 0.5)

# B. The Reliable Shift Bonus (Fixes Match 19/26)
reliable_shift_mask = (pos_df['goals_conceded'] == 1) & (pos_df['xgp'] > 0.0) & (pos_df['save_success_rate'] >= 80.0)
pos_df['raw_score'] += np.where(reliable_shift_mask, 0.25, 0.0)

# ---------------------------------------------------------
# 4. Inverted Clean Sheet Bonuses (Unchanged)
# ---------------------------------------------------------

# 5. The Inverted Tiered Clean Sheet Bonuses
non_pen_goals = pos_df['goals_conceded'] - pos_df['penalty_goals_conceded']
clean_sheet = non_pen_goals == 0

# Clutch Exemption: If xGA <= 1.0 but they mathematically erased ~1.0 expected goal on their own, upgrade to Standard.
passenger_mask = clean_sheet & (pos_df['xg_against'] <= 1.0) & (pos_df['xgp'] < 0.95)
bailout_mask = clean_sheet & (pos_df['xg_against'] >= 2.0) & (pos_df['saves'] >= 5)
standard_cs_mask = clean_sheet & ~passenger_mask & ~bailout_mask

pos_df['raw_score'] += np.where(passenger_mask, 0.30, 0)
pos_df['raw_score'] += np.where(standard_cs_mask, 0.50, 0)
pos_df['raw_score'] += np.where(bailout_mask, 0.80, 0)

In [587]:
# Match supremacy scalar
gamma = 0.2
delta_xg = (pos_df['team_xg'] + 1) / (pos_df['xg_against'] + 1)
match_supremacy_scalar = gamma * np.log(delta_xg)

# Cap the scalar so a 13.0 xG game doesn't ruin the GK rating
match_supremacy_scalar = np.clip(match_supremacy_scalar, a_min=None, a_max=0.25)

In [588]:
k = 0.85
s_0 = np.log(2/3) / k
# Create a new column, applying the sigmoid transformation to the raw score
pos_df['raw_rating'] = 10 * (1 / (1 + np.exp(-k * (pos_df['raw_score'] - s_0))))

In [589]:
pos_df['final_rating'] = pos_df['raw_rating'] - match_supremacy_scalar

In [590]:
pos_df['final_rating'] = pos_df['final_rating'].apply(lambda x: max(0, min(10, x)))

In [591]:
random_row = pos_df.sample(n=1).iloc[0]
for col in pos_df.columns:
    print(f"{col}: {random_row[col]:.2f},")

match_id: 35.00,
shots_against: 3.00,
shots_on_target: 1.00,
saves: 0.00,
goals_conceded: 1.00,
save_success_rate: 0.00,
punch_saves: 0.00,
rush_saves: 0.00,
penalty_saves: 0.00,
penalty_goals_conceded: 0.00,
shoot_out_saves: 0.00,
shoot_out_goals_conceded: 0.00,
player_id: 2.00,
half_length: 10.00,
team_xg: 1.20,
xg_against: 0.60,
xgp: -0.40,
gk_heuristic: -0.60,
raw_score: -0.50,
raw_rating: 4.95,
final_rating: 4.89,


In [592]:
# Print out the column with the worst final rating
worst_row = pos_df.loc[pos_df['final_rating'].idxmin()]
print("\nWorst Final Rating:")
for col in pos_df.columns:
    print(f"{col}: {worst_row[col]:.2f},")


Worst Final Rating:
match_id: 70.00,
shots_against: 10.00,
shots_on_target: 8.00,
saves: 4.00,
goals_conceded: 4.00,
save_success_rate: 50.00,
punch_saves: 0.00,
rush_saves: 0.00,
penalty_saves: 0.00,
penalty_goals_conceded: 0.00,
shoot_out_saves: 0.00,
shoot_out_goals_conceded: 0.00,
player_id: 2.00,
half_length: 10.00,
team_xg: 3.70,
xg_against: 2.00,
xgp: -2.00,
gk_heuristic: -2.20,
raw_score: -1.25,
raw_rating: 3.41,
final_rating: 3.32,
